# K_08 – Alternative Speicher

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Cyril Saladin | **Datum:** März 2026

---

*Positionierung alternativer Speichertechnologien im CH-Marktkontext.*


| [← K_07 – Technologievergleich](K_07_Technologievergleich.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_09 – Eigenverbrauchsoptimierung →](K_09_Eigenverbrauch.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_08'></a>

1. [0. Setup](#0-setup_K_08)
2. [1. Motivation: Warum alternative Speicher?](#1-motivation-warum-alternative-speicher_K_08)
3. [2. Steckbriefe](#2-steckbriefe_K_08)
4. [3. Positionierung: Entladezeit vs. Wirkungsgrad](#3-positionierung-entladezeit-vs-wirkungsgrad_K_08)
5. [4. Einordnung im CH-Kontext](#4-einordnung-im-ch-kontext_K_08)


## Initialisierung<a id='initialisierung_K_08'></a><a id='0-setup_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')
from datetime import datetime

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [ ]:
with open('../sync/config.json') as f:
    CFG = json.load(f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
CHARTS_DIR = os.path.join('../output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: ../sync/config.json
# ── Farben & Stil aus ../sync/config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

print(f"Setup OK | Charts → {CHARTS_DIR}")

---
## 1. Motivation: Warum alternative Speicher? <a id='1-motivation-warum-alternative-speicher_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

Batterien dominieren den Kleinst- und Mittelsegmentmarkt (< 10 MWh).
Für grosse Energiemengen (> 100 MWh) und lange Entladezeiten (> 8 h) sind
mechanische und chemische Speicher oft günstiger und dauerhafter.

In der Schweiz ist **Pumpspeicher** bereits etabliert — er ist mit Abstand
der grösste Energiespeicher des Landes und unverzichtbar für die Stabilität
des europäischen Verbundnetzes.


---
## 2. Steckbriefe <a id='2-steckbriefe_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

### 2.1 Pumpspeicher (Pumped Hydro Storage — PHS)

| Kennwert | Wert |
|---|---|
| **Typische Kapazität** | 100 MWh … 10 GWh |
| **System-CAPEX** | 5–15 EUR/kWh (bei grosser Kapazität) |
| **Lebensdauer** | 50–100 Jahre |
| **Round-Trip-Wirkungsgrad** | 70–85 % |
| **Selbstentladung** | praktisch 0 |
| **Reaktionszeit** | 1–5 Minuten (Anlauf) |
| **CH-Relevanz** | ★★★★★ — wichtigster CH-Speicher |

Überschussstrom pumpt Wasser in ein Oberbecken. Bei Bedarf fliesst es durch
Turbinen zurück. Die Schweiz betreibt ~9.5 GW installierte Pumpspeicherleistung
(weltweit Spitzenposition pro Kopf).

*CH-Beispiele: Nant de Drance (900 MW, VS) · Linth-Limmern (1 GW, GL) · Grimsel KWO (700 MW, BE)*

**Grenzen:** Geographisch gebunden (Topographie), lange Planungs-/Bauzeit,
politische Hürden. Keine dezentrale Verteilung möglich.

---

### 2.2 Druckluftspeicher (CAES — Compressed Air Energy Storage)

| Kennwert | Wert |
|---|---|
| **Typische Kapazität** | 100 MWh … 5 GWh |
| **System-CAPEX** | 50–100 EUR/kWh |
| **Lebensdauer** | 30–40 Jahre |
| **Round-Trip-Wirkungsgrad** | 40–70 % (A-[CAES](../organisation/O_02_Glossar.ipynb#g-caes): bis 70 %) |
| **Reaktionszeit** | Minuten |
| **CH-Relevanz** | ★★ — Geologie ungeeignet für klassisches CAES |

Überschussstrom komprimiert Luft in Kavernen oder Druckbehälter; zur
Stromerzeugung expandiert sie durch Turbinen.

- **A-CAES (adiabat):** Kompressionswärme wird gespeichert und zurückgeführt → [RTE](../organisation/O_02_Glossar.ipynb#g-rte) bis 70 %, kein Gas nötig
- **D-CAES (diabat):** Benötigt Gas zur Nacherhitzung → RTE ~55 %

**Grenzen in CH:** Keine natürlichen Salzkavernen (wie in D/US); Felskavernen teuer.
Eher für Norddeutschland/Nordsee-Raum relevant.

---

### 2.3 Power-to-X (P2X)

| Kennwert | Wert |
|---|---|
| **Varianten** | Power-to-H₂, Power-to-CH₄, Power-to-Heat, Power-to-Liquid |
| **CAPEX Elektrolyseur** | 500–1 000 EUR/kW_el |
| **Wirkungsgrad Elektrolyse** | 60–80 % |
| **Gesamt-RTE (Strom→Strom)** | 25–45 % |
| **Speicherdauer** | Tage … saisonal |
| **CH-Relevanz** | ★★★ — Pilotprojekte aktiv, saisonal interessant |

Überschussstrom (z.B. Sommer-Solar) wird zu H₂ oder synthetischem Methan
und im Gasnetz gespeichert. Für saisonalen Ausgleich (Solar-Sommer → Winter-Heizung)
strategisch interessant; für tägliche Arbitrage wegen sehr niedrigem RTE ungeeignet.

*CH-Projekte: Hybridwerk Aarmatt (Repower, BE) · H2 Volketswil (ZH) · HyFly (Region Zürich-Flughafen)*

---

### 2.4 Schwungradspeicher (Flywheel Energy Storage — FES)

| Kennwert | Wert |
|---|---|
| **Kapazität Einzeleinheit** | 25–125 kWh (moderne Komposit-Rotoren) |
| **Kapazität Array/Farm** | 1–80 MWh (modular kombiniert) |
| **CAPEX** | 1 000–5 000 EUR/kWh (Einzeleinheit) |
| **Lebensdauer** | > 20 Jahre, > 1 000 000 Zyklen |
| **Round-Trip-Wirkungsgrad** | 90–95 % (Magnetlagerung, Vakuum) |
| **Selbstentladung** | 2–20 %/h (stark systemabhängig) |
| **CH-Relevanz** | ★★ — Grid-[FCR](../organisation/O_02_Glossar.ipynb#g-fcr), USV, Militär-Mikronetze |

Einzeleinheiten haben hohe Selbstentladung (≥5 %/h mit mechanischen Lagern, <1 %/Tag
bei Supraleitungs-Magnetlagerung) — tägliche Arbitrage unwirtschaftlich.
Stärken: Millisekunden-Reaktionszeit, praktisch unbegrenzte Zyklenanzahl, temperaturunabhängig.

**Grossskalige Anlagen (Beispiele):**
- Beacon Power, Stephentown NY: 20 MW / 5 MWh (200 Einheiten, Frequenzregelung)
- Amber Kinetics / PG&E, Fresno CA: 20 MW / 80 MWh (4h Entladezeit)
- China: 30 MW Gridanlage, in Betrieb seit 2024

**Militärische Anwendungen:**
Schwungräder eignen sich für Puls-Energiespeicherung bei Hochenergie-Systemen
(z.B. elektromagnetische Startanlagen EMALS auf Flugzeugträgern, Railgun-Energieversorgung).
Vorteile: keine [Degradation](../organisation/O_02_Glossar.ipynb#g-degradation), keine Temperaturempfindlichkeit, kein Brandrisiko —
wichtig in geschlossenen militärischen Plattformen (Schiffe, U-Boote, Fahrzeuge).


---
## 3. Positionierung: Entladezeit vs. Wirkungsgrad <a id='3-positionierung-entladezeit-vs-wirkungsgrad_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

Bubble-Chart: X = Entladezeit [h], Y = Wirkungsgrad [%], Grösse = [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) —
zeigt das technologische Trade-off zwischen Energiedichte und Kosten.


In [ ]:
# ── Bubble-Chart: Speichertechnologien ───────────────────────────────────────
# x = typische Entladezeit [h], y = RTE [%], Bubble-Grösse ~ CAPEX EUR/kWh
names    = ['Li-Ion (LFP)', 'Redox-Flow',  'Pumpspeicher', 'CAES (adiabat)', 'Power-to-X', 'Schwungrad']
dh_typ   = [4,             8,             10,              6,                100,           0.25]
rte      = [93,            75,            80,              65,                35,            92]
capex_n  = [225,           600,           10,              75,               400,           2500]  # EUR/kWh
colors_b = [C_PRIV,C_LOAD,C_CYAN,C_PRICE,C_DISPATCH,C_UTIL]

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors=C_TICK)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

for name, dh, r, cx, col in zip(names, dh_typ, rte, capex_n, colors_b):
    size = max(80, min(2800, cx * 2.2))
    ax.scatter(dh, r, s=size, color=col, alpha=0.75, edgecolors='white', lw=LW_DUENN)
    ax.annotate(f'{name}\n({cx} \u20ac/kWh)', (dh, r),
                textcoords='offset points', xytext=(10, 4),
                color='white', fontsize=8.5)

ax.axhline(80, color='#555', lw=LW_DUENN, ls=':', alpha=0.7)
ax.text(0.06, 81, 'RTE-Schwelle 80 %', color='#666', fontsize=FS_LEGENDE)
ax.set_xscale('log')
ax.set_xlabel('Typische Entladezeit [h]  (log-Skala)', color=C_ACHSE, fontsize=11)
ax.set_ylabel('Round-Trip-Wirkungsgrad [%]', color=C_ACHSE, fontsize=11)
ax.set_title('Speichertechnologien — Entladezeit vs. Wirkungsgrad\n(Bubble-Grösse \u221d System-CAPEX [EUR/kWh])',
             color='white', fontweight='bold')
ax.grid(True, alpha=ALPHA_FLAECHE)
ax.set_xlim(0.02, 300)

p = os.path.join(CHARTS_DIR, 'kuer_k08_positionierung.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")


**Lesehilfe** (log-Skala X-Achse):
- **Oben Mitte/Rechts** — Langer Speicher + hoher Wirkungsgrad: [Li-Ion](../organisation/O_02_Glossar.ipynb#g-li-ion) (4h Utility-Standard) und Pumpspeicher — beide kommerziell etabliert
- **Oben Links** — Sehr kurze Entladezeit + hoher Wirkungsgrad: Schwungrad-Farmen (FCR, 15 min) — Nische Frequenzregelung
- **Mitte** — Mittlere Entladezeit, mittlerer Wirkungsgrad: [Redox-Flow](../organisation/O_02_Glossar.ipynb#g-redox-flow) (8h) und [CAES](../organisation/O_02_Glossar.ipynb#g-caes) (6h) — Utility-Grossspeicher
- **Rechts Unten** — Sehr langer Speicher, niedriger Wirkungsgrad: [Power-to-X](../organisation/O_02_Glossar.ipynb#g-power-to-x) (Tage–saisonal) — Sektorkopplung, nicht für Arbitrage

> **Hinweis Li-Ion:** Der Datenpunkt zeigt 4h (NREL ATB Utility-Standard). Li-Ion deckt tatsächlich 1–10h ab — von Residential (1–2h) bis Utility Long-Duration (8–10h). Die Entladezeit ist konfigurierbar, nicht technologiebedingt fix.

> **Hinweis Schwungrad:** Einzeleinheiten laden in Sekunden, Farmen erreichen 15–60 min effektive Entladezeit. Kapazitäten bis 80 MWh möglich (Arrays).


---
## 4. Einordnung im CH-Kontext <a id='4-einordnung-im-ch-kontext_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

| Technologie | Rol im CH-Energiesystem | Arbitrage-Eignung |
|---|---|---|
| **Pumpspeicher** | Hauptspeicher für saisonalen Ausgleich, Regelenergie | Gut (Grossskala) |
| **Li-Ion (LFP)** | Dezentral, schnell reagierend, modular | Sehr gut (alle Segmente) |
| **Redox-Flow** | Utility-Grossspeicher, langer Zyklushorizont | Gut (> 10 MWh) |
| **CAES** | Derzeit nicht relevant in CH (Geologie) | Bedingt |
| **Power-to-X** | Saisonaler Speicher, Sektorkopplung Strom/Gas/Wärme | Nicht für Arbitrage |
| **Schwungrad** | Netzregelung (FCR), USV — kein Energiespeicher im eigentlichen Sinn | Nein |

**Fazit für das Projekt:**  
[Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) mit Batterien und Pumpspeichern sind **komplementär** —
nicht kompetitiv. Batterien agieren im Tages- und Wochenspeicher-Bereich,
Pumpspeicher sichern den saisonalen Ausgleich und die Frequenzstabilität.
[Power-to-X](../organisation/O_02_Glossar.ipynb#g-power-to-x) schliesst die Lücke für mehrmonatige Speicherung.


---

In [ ]:
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
charts_out = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_k08_')]
print("=== Abschlusskontrolle K_08 (Kür) ===")
for f in sorted(charts_out):
    print(f"  ✅  {f}")
print(f"  {len(charts_out)} Chart(s) erzeugt")
print("=== Ende ===")



---
## Fazit <a id='fazit_K_08'></a>

[↑ Inhaltsverzeichnis](#toc_K_08)

**Batteriespeicher und Pumpspeicher sind komplementär** — kein Wettbewerb.

| Dimension | Li-Ion Batterie | Pumpspeicher (CH) |
|-----------|----------------|-------------------|
| **Zeithorizont** | Stunden bis Tage | Tage bis saisonal |
| **Reaktionszeit** | Millisekunden | 1–5 Minuten |
| **Standort** | Dezentral, überall | Geographisch gebunden (Alpen) |
| **Arbitrage** | Täglich (Intraday) | Saisonal (Sommer pumpen, Winter erzeugen) |
| **CAPEX** | Sinkend (~−10%/J) | Stabil hoch (Infrastruktur) |
| **CH-Relevanz** | Wächst | Bereits ~9.5 GW installiert |

**Für Grid-Arbitrage im Schweizer Kontext:**
- **Privat/Gewerbe/Industrie:** Li-Ion (LFP) dominiert — Module, Kosten, Sicherheit
- **Utility ≥ 10 MWh:** Redox-Flow als strategische Alternative (> 20'000 Zyklen, kein Brandrisiko)
- **Saisonaler Ausgleich:** Bleibt Domäne der Pumpspeicher — keine realistische Batterie-Alternative bei diesen Zeitskalen und Volumina

→ Detaillierter Technologie- und Kostenvergleich: [K_07 – Technologievergleich](K_07_Technologievergleich.ipynb)


---

| [← K_07 – Technologievergleich](K_07_Technologievergleich.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_09 – Eigenverbrauchsoptimierung →](K_09_Eigenverbrauch.ipynb) |
|:---|:---:|---:|